# Tutorial 52: General Creation of Element Dataframes

This Example demonstrates the capabilities of the class Dataframes_SIR3S_Model that extends SIR3S_Model be abilities to work directley with pandas dataframes. It is shown how to create dataframes containing information about elements such as Nodes, Pipes, etc. existing in a SIR 3S Model. The methods presented are not user-defined and neither efficient, but get you the most important information quickly. For more detailed methods of creating dataframes, see Tutorial 51.

# Toolkit Release

In [1]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-06-08 10:40:53,714] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-08 10:40:53,714] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-08 10:40:53,799] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

[2026-06-08 10:41:05,122] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


## Additional

In [8]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx

# Open Model

In [9]:
s3s.OpenModel(dbName=r"Toolkit_Tutorial52_Model.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-06-08 10:41:18,758] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Calculate Model

In [10]:
#s3s.ExecCalculation(True)

# Generate Element Dataframes

We can use the [generate_element_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_dataframe) method to quickly generate basic dataframes containing all instances of hydraulic element types (Node, Pipe, etc.) in a SIR 3S model. 

All model_data and result values (self.GetResultProperties_from_elementType(onlySelectedVectors=False)) for the static timestamp are included. Result values are given as floats.

The pd.Dataframe will automatically be transformed into a gpd.GeoDataFrame if a SRID is defined in the model, after a geometry column is created.

In [11]:
object_types = [item for item in dir(s3s.ObjectTypes) if not (item.startswith('__') and item.endswith('__'))]
print(object_types) # Check for hydraulic elmement types

['AGSN_HydraulicProfile', 'AirVessel', 'Arrow', 'Atmosphere', 'BlockConnectionNode', 'CalcPari', 'CharacteristicLossTable', 'CharacteristicLossTable_Row', 'Circle', 'Compressor', 'CompressorTable', 'CompressorTable_Row', 'ControlEngineeringNexus', 'ControlMode', 'ControlPointTable', 'ControlPointTable_Row', 'ControlValve', 'ControlVariableConverter', 'ControlVariableConverterRSTE', 'CrossSectionTable', 'CrossSectionTable_Row', 'DPGR_DPKT_DatapointDpgrConnection', 'DPGR_DataPointGroup', 'DPKT_Datapoint', 'DamageRatesTable', 'DamageRatesTable_Row', 'DeadTimeElement', 'Demand', 'DifferentialRegulator', 'DirectionalArrow', 'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Divider', 'DriveEfficiencyTable', 'DriveEfficiencyTable_Row', 'DrivePowerTable', 'DrivePowerTable_Row', 'EBES_FeederGroups', 'EfficiencyConverterTable', 'EfficiencyConverterTable_Row', 'ElementQuery', 'EnergyRecoveryTable', 'EnergyRecoveryTable_Row', 'EnvironmentTemp', 'FWBZ_DistrictHeatingReferenceValues', 'FlapValve'

This function allows for little user definition the only paramters are element_type and tks of that element type to exclusively use. For more user defined dataframe creation see Tutorial 51.

## Node

We can parametrize the function to only include certain tks of the element type in the dataframe.

In [12]:
good_tks = []

In [13]:
for tk in s3s.GetTksofElementType(s3s.ObjectTypes.Node):
    if s3s.GetValue(tk, "FkCont") == s3s.GetMainContainer()[0]: # Main container
        good_tks.append(tk)

In [14]:
df_node, timestamp_to_tuple_index = s3s.generate_element_dataframe(element_type=s3s.ObjectTypes.Node, tks=None)
df_node.head(3)

[2026-06-08 10:41:19,005] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df for element type: ObjectTypes.Node ...
[2026-06-08 10:41:19,005] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df_model_data for element type: ObjectTypes.Node ...
[2026-06-08 10:41:19,264] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node
[2026-06-08 10:41:19,267] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 517 element(s) of element type ObjectTypes.Node.
[2026-06-08 10:41:19,292] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Node.
[2026-06-08 10:41:19,292] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 37 model_data properties.
[2026-06-08 10:41:19,292] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data 

,tk,Name,Ktyp,Zkor,QmEin,Lfakt,Fkpzon,Fkfstf,Fkutmp,Fkfqps,Fkcont,Fk2lknot,Beschreibung,Idreferenz,Iplanung,Kvr,Qakt,Xkor,Ykor,NodeNamePosition,ShowNodeName,KvrKlartext,NumberOfVERB,HasBlockConnection,Tk,Pk,InVariant,GeometriesDiffer,SymbolFactor,bz.Drakonz,bz.Fk,bz.Fkpvar,bz.Fkqvar,bz.Fklfkt,bz.PhEin,bz.Tm,bz.Te,bz.PhMin,geometry,BCIND,DP,DPH,H,HMAX_INST,HMIN_INST,IAKTIV,LFAKTAKT,P,PDAMPF,PH,PHMINMAXDIF,PH_EIN,PH_MIN,PMAX_INST,PMIN_INST,QM,RHO,T,TTR,VOLD
0,4612618373909997110,V-K2133S,QKON,543.56,0,1,5520728169779652386,4798673252636751115,5591325053703727727,-1,5029128874972463118,5611267768413515094,Anfangsknoten generiert von SirDB,3S96AE619D388EA6F4CC9F24456148E088,1,1,0,714332.858074,5.578924e+06,1,False,Vorlauf,0,False,4612618373909997110,4612618373909997110,False,False,0.2,0,4612618373909997110,-1,-1,-1,0,0,0,0,POINT (714332.858 5578924.328),"(17.0,)","(0.007866541,)","(0.007866541,)","(1.121148,)","(1.121148,)","(1.121148,)","(1.0,)","(1.0,)","(1.772032,)","(0.0123,)","(0.7720316,)","(0.0,)","(0.7720316,)","(0.0,)","(1.772032,)","(1.772032,)","(0.0,)","(1000.3,)","(10.0,)","(87.79269,)","(0.0,)"
1,4619205996903908050,V-K983S,QKON,548.26,0,1,5520728169779652386,4798673252636751115,5591325053703727727,-1,5029128874972463118,5130743098019975840,Anfangsknoten generiert von SirDB,3S56C0B9A1652EF8E9B7AB8C5DEACA2DC4,1,1,0,713611.070733,5.578598e+06,1,False,Vorlauf,0,False,4619205996903908050,4619205996903908050,False,False,0.2,0,4619205996903908050,-1,-1,-1,0,0,0,0,POINT (713611.071 5578598.067),"(17.0,)","(1.650535,)","(1.650535,)","(4.942434,)","(4.942434,)","(4.942434,)","(0.0,)","(1.0,)","(5.132404,)","(0.6955614,)","(4.132404,)","(0.0,)","(4.132404,)","(0.0,)","(5.132404,)","(5.132404,)","(0.0,)","(965.8268,)","(89.7886,)","(0.2137029,)","(0.0,)"
2,4619682681341516951,R-K2803S,QKON,554.99,0,1,5520728169779652386,4798673252636751115,5591325053703727727,-1,5029128874972463118,5367059433340055050,Anfangsknoten generiert von SirDB,3S7BD49C919428AD75D6EA1203195E860E,1,2,0,713542.639481,5.578805e+06,1,False,Rücklauf,0,False,4619682681341516951,4619682681341516951,False,False,0.2,0,4619682681341516951,-1,-1,-1,0,0,0,0,POINT (713542.639 5578804.842),"(17.0,)","(1.542218,)","(1.542218,)","(3.360077,)","(3.360077,)","(3.360077,)","(0.0,)","(1.0,)","(2.890061,)","(0.1971269,)","(1.890061,)","(0.0,)","(1.890061,)","(0.0,)","(2.890061,)","(2.890061,)","(0.0,)","(983.8152,)","(59.76965,)","(0.1680818,)","(0.0,)"


All results properties are written as tuples. 

In [15]:
df_node["DP"].iloc[0][0]

0.007866541

## Pipe

We can also request a list of timestamps (int or str) to be included in the results.

In [16]:
s3s.GetTimeStamps()[0]

['2023-02-13 00:00:00.000 +01:00',
 '2023-02-13 01:00:00.000 +01:00',
 '2023-02-13 02:00:00.000 +01:00',
 '2023-02-13 03:00:00.000 +01:00',
 '2023-02-13 04:00:00.000 +01:00',
 '2023-02-13 05:00:00.000 +01:00',
 '2023-02-13 06:00:00.000 +01:00',
 '2023-02-13 07:00:00.000 +01:00',
 '2023-02-13 08:00:00.000 +01:00',
 '2023-02-13 09:00:00.000 +01:00',
 '2023-02-13 10:00:00.000 +01:00',
 '2023-02-13 11:00:00.000 +01:00',
 '2023-02-13 12:00:00.000 +01:00',
 '2023-02-13 13:00:00.000 +01:00',
 '2023-02-13 14:00:00.000 +01:00',
 '2023-02-13 15:00:00.000 +01:00',
 '2023-02-13 16:00:00.000 +01:00',
 '2023-02-13 17:00:00.000 +01:00',
 '2023-02-13 18:00:00.000 +01:00',
 '2023-02-13 19:00:00.000 +01:00',
 '2023-02-13 20:00:00.000 +01:00',
 '2023-02-13 21:00:00.000 +01:00',
 '2023-02-13 22:00:00.000 +01:00',
 '2023-02-13 23:00:00.000 +01:00',
 '2023-02-14 00:00:00.000 +01:00']

Let's include the first three timestamps.

In [17]:
df_pipe, timestamp_to_tuple_index = s3s.generate_element_dataframe(element_type=s3s.ObjectTypes.Pipe
                                                                    ,timestamps=[0, 1, 2])
                                

[2026-06-08 10:42:15,810] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df for element type: ObjectTypes.Pipe ...
[2026-06-08 10:42:15,811] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df_model_data for element type: ObjectTypes.Pipe ...
[2026-06-08 10:42:15,917] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:42:15,920] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:42:15,924] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Pipe.
[2026-06-08 10:42:15,925] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 46 model_data properties.
[2026-06-08 10:42:15,926] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data 

In [18]:
df_pipe.head()

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,MVEC_sequence,MVEC_start,PDAMPF,PHR,PMIN,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMAX_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,PVECMIN_INST_start,PVEC_end,PVEC_sequence,PVEC_start,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_end,RHOVEC_sequence,RHOVEC_start,SVEC_end,SVEC_sequence,SVEC_start,TI,TK,TTRVEC_end,TTRVEC_sequence,TTRVEC_start,TVEC_end,TVEC_sequence,TVEC_start,VAV,VI,VK,VOLDA,WVL,ZVEC_end,ZVEC_sequence,ZVEC_start
0,4614463970292122863,Rohr R-K4383S R-K4183S,4689226368751411179,4779752876656844188,5431845028903382031,7.780674,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,4713734746689397424,OSM: Knoten 450994211 -> Knoten 476971188; Län...,166815824,5,2,0.005,0,999,994.0,Rücklauf,,4614463970292122863,4614463970292122863,False,714262.482930,5.578857e+06,False,4614463970292122863,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (714262.483 5578857.42, 714269.543 ...",4730066059089961857,4917189080965035120,KMR,DN 999,4927972999088213410,999,994.0,1014.0,10.0,0,0,1.212,0,0,0,1950,4689226368751411179,4689226368751411179,False,"(0.0, 0.0, 0.0)","(7.780674, 7.780674, 7.780674)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(1.0, 1.0, 1.0)","(0.0, 0.0, 0.0)","(4.303457e-23, 9.849004e-23, 0.0)","(2.837623e-10, -4.292815e-10, 0.0)","((2.837623e-10, 2.837623e-10), (-4.292815e-10,...","(2.837623e-10, -4.292815e-10, 0.0)","(0.0123, 0.0123, 0.0123)","(3.34838e-25, 7.663189e-25, 0.0)","(1.615057, 1.615057, 1.615057)","(1.68863, 1.68863, 1.68863)","((1.615055, 1.68863), (1.615055, 1.68863), (1....","(1.615055, 1.615055, 1.615055)","(1.68863, 1.68863, 1.68863)","((1.615055, 1.68863), (1.615055, 1.68863), (1....","(1.615055, 1.615055, 1.615055)","(1.68863, 1.68863, 1.68863)","((1.615055, 1.68863), (1.615055, 1.68863), (1....","(1.615055, 1.615055, 1.615055)","(1.021544e-09, -1.545413e-09, 0.0)","(1.021544e-09, -1.545413e-09, 0.0)","(1.021544e-09, -1.545413e-09, 0.0)","(1000.3, 1000.3, 1000.3)","(1000.3, 1000.3, 1000.3)","(1000.3, 1000.3, 1000.3)","((1000.3, 1000.3), (1000.3, 1000.3), (1000.3, ...","(1000.3, 1000.3, 1000.3)","(7.780674, 7.780674, 7.780674)","((0.0, 7.780674), (0.0, 7.780674), (0.0, 7.780...","(0.0, 0.0, 0.0)","(9.999994, 9.999994, 9.999994)","(9.999994, 9.999994, 9.999994)","(249.9251, 159186800.0, 46852560.0)","((242.1444, 249.9251), (159186800.0, 159186800...","(242.1444, 159186800.0, 46852560.0)","(10.0, 10.0, 10.0)","((10.0, 10.0), (10.0, 10.0), (10.0, 10.0))","(10.0, 10.0, 10.0)","(3.655627e-13, -5.530307e-13, 0.0)","(3.655627e-13, -5.530307e-13, 0.0)","(3.655627e-13, -5.530307e-13, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(544.34, 544.34, 544.34)","((545.09, 544.34), (545.09, 544.34), (545.09, ...","(545.09, 545.09, 545.09)"
1,4615723899944629797,Rohr V-K203S V-K213S,4689226368751411179,4779752876656844188,5728726059620036726,64.287240,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,4938076287810941486,OSM: Knoten 390310977 -> Knoten 1368674233; Lä...,24633100,5,1,0.005,0,999,994.0,Vorlauf,,4615723899944629797,4615723899944629797,False,713738.296567,5.579220e+06,False,4615723899944629797,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713738.297 5579219.902, 713793.23 ...",5129584372458662150,5332825919690090061,KMR,DN 999,4927972999088213410,999,994.0,1014.0,10.0,0,0,1.212,0,0,0,1950,4689226368751411179,46892

Each result property is written as tuple, where each entry corresponds to one timestamp.

In [19]:
df_pipe["VAV"].iloc[0]

(3.655627e-13, -5.530307e-13, 0.0)

We can use the returned mapping for easy access via literal timestamp.

In [20]:
timestamp_to_tuple_index

{'2023-02-13 00:00:00.000 +01:00': 0,
 '2023-02-13 01:00:00.000 +01:00': 1,
 '2023-02-13 02:00:00.000 +01:00': 2}

In [21]:
df_pipe["VAV"].iloc[0][timestamp_to_tuple_index["2023-02-13 01:00:00.000 +01:00"]]

-5.530307e-13

For already vectorized data (pipe interior points) we have a tuple of tuples with the time tuple as the higher level one.

In [22]:
df_pipe["TTRVEC_sequence"].iloc[0]

((242.1444, 249.9251), (159186800.0, 159186800.0), (46852560.0, 46852560.0))

In [23]:
df_pipe["TTRVEC_sequence"].iloc[0][timestamp_to_tuple_index["2023-02-13 01:00:00.000 +01:00"]]

(159186800.0, 159186800.0)

## DistrictHeatingConsumer

We can specify the timestamp we want the result data for either as "2023-02-13 01:00:00.000 +01:00" or 1 meaning the second element in s3s.GetTimeStamps()[0], otherwise static is used.

In [24]:
df_dhc, timestamp_to_tuple_index = s3s.generate_element_dataframe(element_type=s3s.ObjectTypes.DistrictHeatingConsumer, tks=None, timestamps=[1])
df_dhc.head(3)

[2026-06-08 10:44:15,579] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df for element type: ObjectTypes.DistrictHeatingConsumer ...
[2026-06-08 10:44:15,582] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df_model_data for element type: ObjectTypes.DistrictHeatingConsumer ...
[2026-06-08 10:44:15,711] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.DistrictHeatingConsumer
[2026-06-08 10:44:15,713] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 337 element(s) of element type ObjectTypes.DistrictHeatingConsumer.
[2026-06-08 10:44:15,715] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.DistrictHeatingConsumer.
[2026-06-08 10:44:15,715] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 55 model_data properties.
[2026-

,tk,Name,Beschreibung,Ind0,W0,Qm0,Tvl0,Trs0,Lfk,Rho0,Dtmin,Indtr,Trsk,Fktrft,A,B,C,Vtyp,V0,P1soll,Dpvlmin,Fkzep1vl,Tsvl,Zevk,Dphaus,Dprlmin,Fkzep1rl,Tsrl,Imbg,Irfv,Fkcont,Idreferenz,Iplanung,CPM,NumberOfVERB,IndtrKlartext,M0Estimated,W0Estimated,Tk,Pk,InVariant,Xkor,Ykor,ShowDescription,PositionOfDescription,Angle,SymbolFactor,GeometriesDiffer,bz.Fk,bz.Indlast,bz.Indlfkt2,bz.Fklfkt,bz.Fklfkt2,bz.Fkqvar,bz.Fktevt,bz.IndlastKlartext,geometry,fkKI,fkKK,DH,DP,DPH,IAKTIV,INDUV,LFH,LFT,M,MHYUV,MTHUV,PHIRL,PHIVL,QM,RHOI,RHOK,TI,TK,TVMIN,W,WSOLL
0,4611752310942477664,Fernwärmeverbraucher V-K1203S R-K3683S,Gattendorf;95185;Obere Au;28;None;yes,0,139.0433,1,90,60,1,1000,3,3,60,-1,2,-0.25,-2,1,1,0,0,-1,0,0,0.2,0.3,-1,0,0,1,5029128874972463118,830887238,1,4.1903,0,"Tabelle Temperatur TEVT, TRS(t)",3.981862,34.91917,4611752310942477664,4611752310942477664,False,713675.300232,5.578705e+06,False,3,0,0.1,False,4611752310942477664,0,0,4835417738045943522,,-1,5395645951786400348,Lastfaktor thermisch,POINT (713675.3 5578705.193),4673701597187411685,4995788945387711669,"(17.20627,)","(1.58637,)","(1.58637,)","(0.0,)","(0.0,)","(0.2917569,)","(0.2865425,)","(0.3226549,)","(-0.838969,)","(0.0,)","(-3.333333e+32,)","(-3.333333e+32,)","(1.161558,)","(966.0209,)","(983.7,)","(89.46521,)","(60.0,)","(68.60849,)","(39.84182,)","(39.83362,)"
1,4612528660388965271,Fernwärmeverbraucher V-K1783S R-K4263S,None;None;None;None;None;yes,0,386.4134,1,90,60,1,1000,3,3,60,-1,2,-0.25,-2,1,0,0,0,-1,0,0,0.2,0.3,-1,0,0,1,5029128874972463118,1056287317,1,4.1903,0,"Tabelle Temperatur TEVT, TRS(t)",11.065940,34.91917,4612528660388965271,4612528660388965271,False,714251.332395,5.578925e+06,False,3,0,0.1,False,4612528660388965271,0,0,5554262436821166605,,-1,5395645951786400348,Lastfaktor thermisch,POINT (714251.332 5578925.001),5015101891725198603,5751808837348052764,"(0.07000709,)","(0.00686741,)","(0.00686741,)","(1.0,)","(0.0,)","(0.0,)","(0.0,)","(0.0,)","(0.0,)","(0.0,)","(-3.333333e+32,)","(-3.333333e+32,)","(0.0,)","(1000.3,)","(983.7,)","(10.0,)","(60.0,)","(10.0,)","(0.0,)","(0.0,)"
2,4612562908060328263,Fernwärmeverbraucher V-K1623S R-K4103S,Gattendorf;95185;Langenbachstraße;4;None;yes,0,106.6409,1,90,60,1,1000,3,3,60,-1,2,-0.25,-2,1,1,0,0,-1,0,0,0.2,0.3,-1,0,0,1,5029128874972463118,830818182,1,4.1903,0,"Tabelle Temperatur TEVT, TRS(t)",3.053936,34.91917,4612562908060328263,4612562908060328263,False,713271.271817,5.578980e+06,False,3,0,0.1,False,4612562908060328263,0,0,4835417738045943522,,-1,5395645951786400348,Lastfaktor thermisch,POINT (713271.272 5578979.743),4965299629814639205,4779536530687993701,"(16.77219,)","(1.564011,)","(1.564011,)","(0.0,)","(0.0,)","(0.2908092,)","(0.2865336,)","(0.2466601,)","(-0.6413593,)","(0.0,)","(-3.333333e+32,)","(-3.333333e+32,)","(0.8879764,)","(965.9639,)","(983.7,)","(89.56009,)","(60.0,)","(68.60849,)","(30.5562,)","(30.55087,)"
